# BÀI TẬP CHƯƠNG 2 – SQL với Python (SQLite)
Sử dụng SQLite trong Python để thực hiện các thao tác SQL trên hai bảng `student` và `course`.

In [1]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta

# Kết nối SQLite in-memory
conn = sqlite3.connect(':memory:')
cur = conn.cursor()

# Tạo bảng course
cur.execute('''
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
)
''')

# Tạo bảng student
cur.execute('''
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
''')

# Chèn dữ liệu course
cur.executemany('INSERT INTO course VALUES (?, ?)', [
    (12, 'Giai tich'),
    (34, 'Thong ke'),
    (26, 'Tin hoc')
])

# Chèn dữ liệu student (NULL cho course_id bị thiếu)
cur.executemany('INSERT INTO student VALUES (?, ?, ?, ?, ?)', [
    (1,  'Nguyen Minh Hoang', 'May Tinh', 12,   6.7),
    (2,  'Tran Thi Lan',      'Kinh Te',  34,   9.2),
    (3,  'Pham Van Nam',      'Toan Tin', None, 7.9),
    (4,  'Le Thanh Huyen',    'Toan Tin', 20,   7.2),
    (5,  'Vu Quoc Anh',       'May Tinh', 24,   8.0),
    (6,  'Dang Thuy Linh',    'May Tinh', 24,   5.5),
    (7,  'Bui Tien Dung',     'Kinh Te',  34,   9.2),
    (8,  'Ho Ngoc Mai',       'Toan Tin', 20,   8.8),
    (9,  'Duong Huu Phuc',    'Kinh Te',  None, 7.2),
    (10, 'Cao Thi Hanh',      'May Tinh', None, 7.0)
])

conn.commit()
print(' Đã tạo và nhập dữ liệu thành công!')

print('\n Bảng STUDENT:')
display(pd.read_sql('SELECT * FROM student', conn))
print('\n Bảng COURSE:')
display(pd.read_sql('SELECT * FROM course', conn))

 Đã tạo và nhập dữ liệu thành công!

 Bảng STUDENT:


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,9.2
2,3,Pham Van Nam,Toan Tin,NaN,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,7.0



 Bảng COURSE:


,id,course_name
0,12,Giai tich
1,26,Tin hoc
2,34,Thong ke


## Câu 1: Kết nối hai bảng

In [2]:
print('='*70)
print('1a. TÍCH DESCARTES (Cartesian Product)')
print('='*70)
df = pd.read_sql('SELECT * FROM student, course', conn)
print(f'Số dòng kết quả: {len(df)} (= {len(pd.read_sql("SELECT * FROM student",conn))} x {len(pd.read_sql("SELECT * FROM course",conn))})')
display(df)

1a. TÍCH DESCARTES (Cartesian Product)
Số dòng kết quả: 30 (= 10 x 3)


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
8,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


In [3]:
print('='*70)
print('1b. INNER JOIN')
print('='*70)
sql = '''
SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score
FROM student s
INNER JOIN course c ON s.course_id = c.id
'''
df = pd.read_sql(sql, conn)
print(f'Số dòng: {len(df)}')
display(df)
print('Kết luận: INNER JOIN chỉ trả về các sinh viên có course_id khớp với bảng course (loại bỏ course_id NULL và không tồn tại).')

1b. INNER JOIN
Số dòng: 3


,student_id,name,class,course_id,course_name,score
0,1,Nguyen Minh Hoang,May Tinh,12,Giai tich,6.7
1,2,Tran Thi Lan,Kinh Te,34,Thong ke,9.2
2,7,Bui Tien Dung,Kinh Te,34,Thong ke,9.2


Kết luận: INNER JOIN chỉ trả về các sinh viên có course_id khớp với bảng course (loại bỏ course_id NULL và không tồn tại).


In [4]:
print('='*70)
print('1c. LEFT JOIN')
print('='*70)
sql = '''
SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score
FROM student s
LEFT JOIN course c ON s.course_id = c.id
'''
df = pd.read_sql(sql, conn)
print(f'Số dòng: {len(df)}')
display(df)
print(' Kết luận: LEFT JOIN giữ toàn bộ sinh viên, course_name = NULL nếu không có môn học khớp.')

1c. LEFT JOIN
Số dòng: 10


,student_id,name,class,course_id,course_name,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,Giai tich,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,Thong ke,9.2
2,3,Pham Van Nam,Toan Tin,NaN,None,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,None,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,None,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,None,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,Thong ke,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,None,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,None,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,None,7.0


 Kết luận: LEFT JOIN giữ toàn bộ sinh viên, course_name = NULL nếu không có môn học khớp.


In [5]:
print('='*70)
print('1d. RIGHT JOIN (mô phỏng bằng LEFT JOIN đảo bảng)')
print('='*70)
# SQLite không hỗ trợ RIGHT JOIN trực tiếp => đảo chiều LEFT JOIN
sql = '''
SELECT s.student_id, s.name, s.class, s.course_id, c.id AS course_id_from_course, c.course_name, s.score
FROM course c
LEFT JOIN student s ON s.course_id = c.id
'''
df = pd.read_sql(sql, conn)
print(f'Số dòng: {len(df)}')
display(df)
print(' Kết luận: RIGHT JOIN giữ toàn bộ các môn học trong bảng course, sinh viên = NULL nếu không có ai học môn đó (vd: Tin hoc - id 26).')

1d. RIGHT JOIN (mô phỏng bằng LEFT JOIN đảo bảng)
Số dòng: 4


,student_id,name,class,course_id,course_id_from_course,course_name,score
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,12,Giai tich,6.7
1,NaN,None,None,NaN,26,Tin hoc,NaN
2,7.0,Bui Tien Dung,Kinh Te,34.0,34,Thong ke,9.2
3,2.0,Tran Thi Lan,Kinh Te,34.0,34,Thong ke,9.2


 Kết luận: RIGHT JOIN giữ toàn bộ các môn học trong bảng course, sinh viên = NULL nếu không có ai học môn đó (vd: Tin hoc - id 26).


In [6]:
print('='*70)
print('1e. FULL OUTER JOIN (mô phỏng bằng UNION của LEFT JOIN và RIGHT JOIN)')
print('='*70)
sql = '''
SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score
FROM student s
LEFT JOIN course c ON s.course_id = c.id

UNION

SELECT s.student_id, s.name, s.class, c.id, c.course_name, s.score
FROM course c
LEFT JOIN student s ON s.course_id = c.id
'''
df = pd.read_sql(sql, conn)
print(f'Số dòng: {len(df)}')
display(df)
print(' Kết luận: FULL OUTER JOIN trả về toàn bộ dữ liệu từ cả hai bảng, NULL ở những chỗ không khớp.')

1e. FULL OUTER JOIN (mô phỏng bằng UNION của LEFT JOIN và RIGHT JOIN)
Số dòng: 11


,student_id,name,class,course_id,course_name,score
0,NaN,None,None,26.0,Tin hoc,NaN
1,1.0,Nguyen Minh Hoang,May Tinh,12.0,Giai tich,6.7
2,2.0,Tran Thi Lan,Kinh Te,34.0,Thong ke,9.2
3,3.0,Pham Van Nam,Toan Tin,NaN,None,7.9
4,4.0,Le Thanh Huyen,Toan Tin,20.0,None,7.2
5,5.0,Vu Quoc Anh,May Tinh,24.0,None,8.0
6,6.0,Dang Thuy Linh,May Tinh,24.0,None,5.5
7,7.0,Bui Tien Dung,Kinh Te,34.0,Thong ke,9.2
8,8.0,Ho Ngoc Mai,Toan Tin,20.0,None,8.8
9,9.0,Duong Huu Phuc,Kinh Te,NaN,None,7.2


 Kết luận: FULL OUTER JOIN trả về toàn bộ dữ liệu từ cả hai bảng, NULL ở những chỗ không khớp.


## Câu 2: Cập nhật course_id và thống kê

In [7]:
print('='*70)
print('2. Cập nhật course_id còn thiếu')
print('='*70)
print('Phân tích:')
print('- Student 3 (Pham Van Nam): course_id = NULL => Gán course_id = 26 (Tin hoc)')
print('- Student 9 (Duong Huu Phuc): course_id = NULL => Gán course_id = 12 (Giai tich)')
print('- Student 10 (Cao Thi Hanh): course_id = NULL => Gán course_id = 26 (Tin hoc)')
print('- Student 4, 5, 6, 8: course_id = 20 hoặc 24 không tồn tại trong course => sẽ xóa khỏi bảng student')

# Gán course_id hợp lệ cho các NULL
cur.execute("UPDATE student SET course_id = 26 WHERE student_id = 3")
cur.execute("UPDATE student SET course_id = 12 WHERE student_id = 9")
cur.execute("UPDATE student SET course_id = 26 WHERE student_id = 10")

# Xóa các bản ghi có course_id không tồn tại trong bảng course
cur.execute('''
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
''')
conn.commit()

print('\nBảng student sau khi cập nhật và loại bỏ dữ liệu không hợp lệ:')
display(pd.read_sql('SELECT * FROM student', conn))

2. Cập nhật course_id còn thiếu
Phân tích:
- Student 3 (Pham Van Nam): course_id = NULL => Gán course_id = 26 (Tin hoc)
- Student 9 (Duong Huu Phuc): course_id = NULL => Gán course_id = 12 (Giai tich)
- Student 10 (Cao Thi Hanh): course_id = NULL => Gán course_id = 26 (Tin hoc)
- Student 4, 5, 6, 8: course_id = 20 hoặc 24 không tồn tại trong course => sẽ xóa khỏi bảng student

Bảng student sau khi cập nhật và loại bỏ dữ liệu không hợp lệ:


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,26,7.9
3,7,Bui Tien Dung,Kinh Te,34,9.2
4,9,Duong Huu Phuc,Kinh Te,12,7.2
5,10,Cao Thi Hanh,May Tinh,26,7.0


In [8]:
# Sau khi cập nhật và xóa dữ liệu sai, toàn bộ student còn lại đều hợp lệ
sql_valid = '''
SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score
FROM student s
INNER JOIN course c ON s.course_id = c.id
'''
df_valid = pd.read_sql(sql_valid, conn)
print(f' Số sinh viên còn lại sau khi làm sạch dữ liệu: {len(df_valid)} người')
display(df_valid)

 Số sinh viên còn lại sau khi làm sạch dữ liệu: 6 người


,student_id,name,class,course_id,course_name,score
0,1,Nguyen Minh Hoang,May Tinh,12,Giai tich,6.7
1,2,Tran Thi Lan,Kinh Te,34,Thong ke,9.2
2,3,Pham Van Nam,Toan Tin,26,Tin hoc,7.9
3,7,Bui Tien Dung,Kinh Te,34,Thong ke,9.2
4,9,Duong Huu Phuc,Kinh Te,12,Giai tich,7.2
5,10,Cao Thi Hanh,May Tinh,26,Tin hoc,7.0


In [9]:
print('='*70)
print('2a. Tổng số sinh viên và điểm trung bình theo LỚP')
print('='*70)
sql = '''
SELECT s.class AS Lop,
       COUNT(s.student_id) AS Tong_SV,
       ROUND(AVG(s.score), 2) AS Diem_TB
FROM student s
INNER JOIN course c ON s.course_id = c.id
GROUP BY s.class
ORDER BY s.class
'''
df = pd.read_sql(sql, conn)
display(df)
print(' Kết luận:')
for _, row in df.iterrows():
    print(f'  - Lớp {row["Lop"]}: {int(row["Tong_SV"])} sinh viên, điểm TB = {row["Diem_TB"]}')

2a. Tổng số sinh viên và điểm trung bình theo LỚP


,Lop,Tong_SV,Diem_TB
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


 Kết luận:
  - Lớp Kinh Te: 3 sinh viên, điểm TB = 8.53
  - Lớp May Tinh: 2 sinh viên, điểm TB = 6.85
  - Lớp Toan Tin: 1 sinh viên, điểm TB = 7.9


In [10]:
print('='*70)
print('2b. Tổng số sinh viên và điểm trung bình theo MÔN HỌC')
print('='*70)
sql = '''
SELECT c.course_name AS Mon_Hoc,
       COUNT(s.student_id) AS Tong_SV,
       ROUND(AVG(s.score), 2) AS Diem_TB
FROM student s
INNER JOIN course c ON s.course_id = c.id
GROUP BY c.id, c.course_name
ORDER BY c.course_name
'''
df = pd.read_sql(sql, conn)
display(df)
print(' Kết luận:')
for _, row in df.iterrows():
    print(f'  - Môn {row["Mon_Hoc"]}: {int(row["Tong_SV"])} sinh viên, điểm TB = {row["Diem_TB"]}')

2b. Tổng số sinh viên và điểm trung bình theo MÔN HỌC


,Mon_Hoc,Tong_SV,Diem_TB
0,Giai tich,2,6.95
1,Thong ke,2,9.20
2,Tin hoc,2,7.45


 Kết luận:
  - Môn Giai tich: 2 sinh viên, điểm TB = 6.95
  - Môn Thong ke: 2 sinh viên, điểm TB = 9.2
  - Môn Tin hoc: 2 sinh viên, điểm TB = 7.45


In [11]:
print('='*70)
print('2c. Phân loại thi đua theo điểm TB từng môn học')
print('='*70)
sql = '''
SELECT c.course_name AS Mon_Hoc,
       COUNT(s.student_id) AS Tong_SV,
       ROUND(AVG(s.score), 2) AS Diem_TB,
       CASE
           WHEN AVG(s.score) >= 9.0 THEN 'Xuất sắc'
           WHEN AVG(s.score) >= 6.0 THEN 'Tốt'
           ELSE 'Kém'
       END AS Phan_Loai
FROM student s
INNER JOIN course c ON s.course_id = c.id
GROUP BY c.id, c.course_name
ORDER BY Diem_TB DESC
'''
df = pd.read_sql(sql, conn)
display(df)
print(' Kết luận:')
for _, row in df.iterrows():
    print(f'  - {row["Mon_Hoc"]}: Điểm TB = {row["Diem_TB"]} => {row["Phan_Loai"]}')

2c. Phân loại thi đua theo điểm TB từng môn học


,Mon_Hoc,Tong_SV,Diem_TB,Phan_Loai
0,Thong ke,2,9.20,Xuất sắc
1,Tin hoc,2,7.45,Tốt
2,Giai tich,2,6.95,Tốt


 Kết luận:
  - Thong ke: Điểm TB = 9.2 => Xuất sắc
  - Tin hoc: Điểm TB = 7.45 => Tốt
  - Giai tich: Điểm TB = 6.95 => Tốt


## Câu 3: Xếp hạng sinh viên

In [12]:
print('='*70)
print('3a. Xếp hạng theo ĐIỂM SỐ (toàn bộ)')
print('='*70)
sql = '''
SELECT s.student_id, s.name, s.class, c.course_name, s.score,
       RANK() OVER (ORDER BY s.score DESC) AS Hang
FROM student s
INNER JOIN course c ON s.course_id = c.id
ORDER BY Hang
'''
df = pd.read_sql(sql, conn)
display(df)

top3 = df.head(3)
bot3 = df.tail(3)
print('\n Top 3 cao nhất:')
display(top3[['name','score','Hang']])
print('\n Top 3 thấp nhất:')
display(bot3[['name','score','Hang']])

3a. Xếp hạng theo ĐIỂM SỐ (toàn bộ)


,student_id,name,class,course_name,score,Hang
0,2,Tran Thi Lan,Kinh Te,Thong ke,9.2,1
1,7,Bui Tien Dung,Kinh Te,Thong ke,9.2,1
2,3,Pham Van Nam,Toan Tin,Tin hoc,7.9,3
3,9,Duong Huu Phuc,Kinh Te,Giai tich,7.2,4
4,10,Cao Thi Hanh,May Tinh,Tin hoc,7.0,5
5,1,Nguyen Minh Hoang,May Tinh,Giai tich,6.7,6



 Top 3 cao nhất:


,name,score,Hang
0,Tran Thi Lan,9.2,1
1,Bui Tien Dung,9.2,1
2,Pham Van Nam,7.9,3



 Top 3 thấp nhất:


,name,score,Hang
3,Duong Huu Phuc,7.2,4
4,Cao Thi Hanh,7.0,5
5,Nguyen Minh Hoang,6.7,6


In [13]:
print('='*70)
print('3b. Xếp hạng theo ĐIỂM SỐ trong từng LỚP')
print('='*70)
sql = '''
SELECT s.student_id, s.name, s.class, c.course_name, s.score,
       RANK() OVER (PARTITION BY s.class ORDER BY s.score DESC) AS Hang_Trong_Lop
FROM student s
INNER JOIN course c ON s.course_id = c.id
ORDER BY s.class, Hang_Trong_Lop
'''
df = pd.read_sql(sql, conn)
display(df)

print('\n Top 3 cao nhất theo lớp:')
top3_class = df[df['Hang_Trong_Lop'] <= 3].sort_values(['class','Hang_Trong_Lop'])
display(top3_class[['name','class','score','Hang_Trong_Lop']])

print('\n Top 3 thấp nhất theo lớp:')
bot3_class = df.sort_values(['class','Hang_Trong_Lop'], ascending=[True,False])
bot3_class = bot3_class.groupby('class').head(3).sort_values(['class','Hang_Trong_Lop'], ascending=[True,False])
display(bot3_class[['name','class','score','Hang_Trong_Lop']])

3b. Xếp hạng theo ĐIỂM SỐ trong từng LỚP


,student_id,name,class,course_name,score,Hang_Trong_Lop
0,2,Tran Thi Lan,Kinh Te,Thong ke,9.2,1
1,7,Bui Tien Dung,Kinh Te,Thong ke,9.2,1
2,9,Duong Huu Phuc,Kinh Te,Giai tich,7.2,3
3,10,Cao Thi Hanh,May Tinh,Tin hoc,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,Giai tich,6.7,2
5,3,Pham Van Nam,Toan Tin,Tin hoc,7.9,1



 Top 3 cao nhất theo lớp:


,name,class,score,Hang_Trong_Lop
0,Tran Thi Lan,Kinh Te,9.2,1
1,Bui Tien Dung,Kinh Te,9.2,1
2,Duong Huu Phuc,Kinh Te,7.2,3
3,Cao Thi Hanh,May Tinh,7.0,1
4,Nguyen Minh Hoang,May Tinh,6.7,2
5,Pham Van Nam,Toan Tin,7.9,1



 Top 3 thấp nhất theo lớp:


,name,class,score,Hang_Trong_Lop
2,Duong Huu Phuc,Kinh Te,7.2,3
0,Tran Thi Lan,Kinh Te,9.2,1
1,Bui Tien Dung,Kinh Te,9.2,1
4,Nguyen Minh Hoang,May Tinh,6.7,2
3,Cao Thi Hanh,May Tinh,7.0,1
5,Pham Van Nam,Toan Tin,7.9,1


In [14]:
print('='*70)
print('3c. Xếp hạng theo ĐIỂM SỐ trong từng MÔN HỌC')
print('='*70)
sql = '''
SELECT s.student_id, s.name, s.class, c.course_name, s.score,
       RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) AS Hang_Trong_Mon
FROM student s
INNER JOIN course c ON s.course_id = c.id
ORDER BY c.course_name, Hang_Trong_Mon
'''
df = pd.read_sql(sql, conn)
display(df)

print('\n Top 3 cao nhất theo môn học:')
top3_course = df[df['Hang_Trong_Mon'] <= 3].sort_values(['course_name','Hang_Trong_Mon'])
display(top3_course[['name','course_name','score','Hang_Trong_Mon']])

print('\n Top 3 thấp nhất theo môn học:')
bot3_course = df.sort_values(['course_name','Hang_Trong_Mon'], ascending=[True,False])
bot3_course = bot3_course.groupby('course_name').head(3).sort_values(['course_name','Hang_Trong_Mon'], ascending=[True,False])
display(bot3_course[['name','course_name','score','Hang_Trong_Mon']])

3c. Xếp hạng theo ĐIỂM SỐ trong từng MÔN HỌC


,student_id,name,class,course_name,score,Hang_Trong_Mon
0,9,Duong Huu Phuc,Kinh Te,Giai tich,7.2,1
1,1,Nguyen Minh Hoang,May Tinh,Giai tich,6.7,2
2,2,Tran Thi Lan,Kinh Te,Thong ke,9.2,1
3,7,Bui Tien Dung,Kinh Te,Thong ke,9.2,1
4,3,Pham Van Nam,Toan Tin,Tin hoc,7.9,1
5,10,Cao Thi Hanh,May Tinh,Tin hoc,7.0,2



 Top 3 cao nhất theo môn học:


,name,course_name,score,Hang_Trong_Mon
0,Duong Huu Phuc,Giai tich,7.2,1
1,Nguyen Minh Hoang,Giai tich,6.7,2
2,Tran Thi Lan,Thong ke,9.2,1
3,Bui Tien Dung,Thong ke,9.2,1
4,Pham Van Nam,Tin hoc,7.9,1
5,Cao Thi Hanh,Tin hoc,7.0,2



 Top 3 thấp nhất theo môn học:


,name,course_name,score,Hang_Trong_Mon
1,Nguyen Minh Hoang,Giai tich,6.7,2
0,Duong Huu Phuc,Giai tich,7.2,1
2,Tran Thi Lan,Thong ke,9.2,1
3,Bui Tien Dung,Thong ke,9.2,1
5,Cao Thi Hanh,Tin hoc,7.0,2
4,Pham Van Nam,Tin hoc,7.9,1


## Câu 4: Thêm trường graduation_date

In [15]:
print('='*70)
print('4. Thêm trường graduation_date vào bảng student')
print('='*70)

# Thêm cột graduation_date kiểu DATETIME
cur.execute('ALTER TABLE student ADD COLUMN graduation_date DATETIME')
conn.commit()

# Lấy xếp hạng của từng sinh viên còn lại theo điểm số (toàn bảng)
sql_rank = '''
SELECT student_id,
       RANK() OVER (ORDER BY score DESC) AS hang
FROM student
'''
df_rank = pd.read_sql(sql_rank, conn)

# Thời gian hiện tại
now = datetime.now()
print(f'Thời gian hiện tại: {now.strftime("%Y-%m-%d %H:%M:%S")}')

# Cập nhật graduation_date = now + hang (ngày)
for _, row in df_rank.iterrows():
    grad_date = (now + timedelta(days=int(row['hang']))).strftime('%Y-%m-%d %H:%M:%S')
    cur.execute(
        'UPDATE student SET graduation_date = ? WHERE student_id = ?',
        (grad_date, int(row['student_id']))
    )
conn.commit()

print('\n Bảng student sau khi thêm graduation_date:')
sql_result = '''
SELECT s.student_id, s.name, s.class, c.course_name, s.score,
       RANK() OVER (ORDER BY s.score DESC) AS Hang,
       s.graduation_date
FROM student s
INNER JOIN course c ON s.course_id = c.id
ORDER BY Hang
'''
df_final = pd.read_sql(sql_result, conn)
display(df_final)
print(' Kết luận: graduation_date được xác định bằng thời điểm hiện tại cộng với số hạng theo điểm số của từng sinh viên sau khi làm sạch dữ liệu.')

4. Thêm trường graduation_date vào bảng student
Thời gian hiện tại: 2026-04-02 20:17:21

 Bảng student sau khi thêm graduation_date:


,student_id,name,class,course_name,score,Hang,graduation_date
0,2,Tran Thi Lan,Kinh Te,Thong ke,9.2,1,2026-04-03 20:17:21
1,7,Bui Tien Dung,Kinh Te,Thong ke,9.2,1,2026-04-03 20:17:21
2,3,Pham Van Nam,Toan Tin,Tin hoc,7.9,3,2026-04-05 20:17:21
3,9,Duong Huu Phuc,Kinh Te,Giai tich,7.2,4,2026-04-06 20:17:21
4,10,Cao Thi Hanh,May Tinh,Tin hoc,7.0,5,2026-04-07 20:17:21
5,1,Nguyen Minh Hoang,May Tinh,Giai tich,6.7,6,2026-04-08 20:17:21


 Kết luận: graduation_date được xác định bằng thời điểm hiện tại cộng với số hạng theo điểm số của từng sinh viên sau khi làm sạch dữ liệu.


In [16]:
conn.close()
print(' Hoàn thành tất cả các câu bài tập Chương 2!')

 Hoàn thành tất cả các câu bài tập Chương 2!
